# Data Preprocessing
Load the raw seismic data from `data/raw`, clean it, and save the processed file into `data/processed`.


In [12]:
%matplotlib inline
import os
import pandas as pd
import numpy as np

raw_path = '../data/raw/seismic-bumps.csv'
processed_dir = '../data/processed'
processed_path = os.path.join(processed_dir, 'cleaned_seismic_data.csv')
categorical_mappings_path = os.path.join(processed_dir, 'categorical_mappings.csv')

os.makedirs(processed_dir, exist_ok=True)

# Load raw data
df = pd.read_csv(raw_path)
print(f'Loaded raw data with shape: {df.shape}')
print(f'\nInitial class distribution:')
print(df['class'].value_counts())


Loaded raw data with shape: (2584, 19)

Initial class distribution:
class
0    2414
1     170
Name: count, dtype: int64


In [13]:

# 1. Handle duplicates (there are a few exact duplicates in the file)
initial_rows = len(df)
df = df.drop_duplicates()
print(f'Removed {initial_rows - len(df)} duplicate rows')


Removed 6 duplicate rows


In [14]:

# 2. Verify no NaN values exist (we know the data is complete)
if df.isna().sum().sum() > 0:
    print(f'Warning: Found {df.isna().sum().sum()} NaN values - investigating...')
    print(df.isna().sum()[df.isna().sum() > 0])
else:
    print('✓ No missing values detected in raw data')


✓ No missing values detected in raw data


In [15]:

# 3. Separate features by type
categorical_cols = ['seismic', 'seismoacoustic', 'shift', 'hazard']
numerical_cols = [col for col in df.columns if col not in categorical_cols + ['class']]
target_col = 'class'


In [16]:

# 4. Create a copy for processing
df_processed = df.copy()


In [17]:

# 5. Manual encoding of categorical variables (no sklearn needed)
print('\nCategorical encoding mappings:')
mappings = []

# Manual label encoding function
def manual_label_encode(series, column_name):
    unique_values = series.unique()
    # Sort to ensure consistent encoding (alphabetically)
    unique_values = sorted(unique_values)
    encoding_map = {val: idx for idx, val in enumerate(unique_values)}
    
    # Store mappings for documentation
    for original, encoded in encoding_map.items():
        mappings.append({'column': column_name, 'original': original, 'encoded': encoded})
    
    print(f'  {column_name}: {encoding_map}')
    return series.map(encoding_map)

# Apply manual encoding
for col in categorical_cols:
    df_processed[col] = manual_label_encode(df_processed[col], col)

# Save mappings to CSV
mappings_df = pd.DataFrame(mappings)
mappings_df.to_csv(categorical_mappings_path, index=False)

# 6. Handle zero-inflation correctly - DO NOT IMPUTE ZEROS
# Create binary indicators for zero-inflated columns to preserve the "no event" signal
zero_inflated_cols = ['nbumps', 'nbumps2', 'nbumps3', 'nbumps4', 'nbumps5', 
                      'nbumps6', 'nbumps7', 'nbumps89', 'energy', 'maxenergy']

for col in zero_inflated_cols:
    if col in df_processed.columns:
        # Create binary indicator: 1 if event occurred, 0 if not
        df_processed[f'{col}_occurred'] = (df_processed[col] > 0).astype(int)

print(f'\n✓ Created binary occurrence indicators for {len(zero_inflated_cols)} zero-inflated columns')

# 7. Add derived seismic features (common in mining seismology research)
# Energy ratio (max/avg) - handles zeros safely
df_processed['energy_ratio'] = np.where(
    df_processed['energy'] > 0,
    df_processed['maxenergy'] / df_processed['energy'],
    0
)

# Seismic moment proxy (energy-based)
df_processed['log_energy'] = np.log1p(df_processed['energy'])  # log(energy + 1) to handle zeros



Categorical encoding mappings:
  seismic: {'a': 0, 'b': 1}
  seismoacoustic: {'a': 0, 'b': 1, 'c': 2}
  shift: {'N': 0, 'W': 1}
  hazard: {'a': 0, 'b': 1, 'c': 2}

✓ Created binary occurrence indicators for 10 zero-inflated columns


In [18]:

# Shift risk indicator (based on domain knowledge)
# Need to map from original dataframe since we encoded it
shift_risk_map = {'W': 1, 'N': 0}
df_processed['shift_risk'] = df['shift'].map(shift_risk_map)

# 8. Validate class distribution after processing
print(f'\nFinal class distribution:')
print(df_processed['class'].value_counts())
print(f'Class imbalance ratio: {df_processed["class"].value_counts()[0] / df_processed["class"].value_counts()[1]:.2f}:1')



Final class distribution:
class
0    2408
1     170
Name: count, dtype: int64
Class imbalance ratio: 14.16:1


In [19]:

# 9. Generate data quality report
print(f'\nData Quality Report:')
print(f'  Total samples: {len(df_processed)}')
print(f'  Features: {len(df_processed.columns)}')
print(f'  Hazardous events (class=1): {df_processed["class"].sum()} ({df_processed["class"].mean()*100:.2f}%)')
print(f'  Zero-energy events: {(df_processed["energy"] == 0).sum()} ({(df_processed["energy"] == 0).mean()*100:.1f}%)')
print(f'  Zero-bump records: {(df_processed["nbumps"] == 0).sum()} ({(df_processed["nbumps"] == 0).mean()*100:.1f}%)')



Data Quality Report:
  Total samples: 2578
  Features: 32
  Hazardous events (class=1): 170 (6.59%)
  Zero-energy events: 1458 (56.6%)
  Zero-bump records: 1458 (56.6%)


In [20]:

# 10. Save processed data
df_processed.to_csv(processed_path, index=False)
print(f'\n✓ Cleaned data saved to: {processed_path}')
print(f'✓ Categorical mappings saved to: {categorical_mappings_path}')



✓ Cleaned data saved to: ../data/processed\cleaned_seismic_data.csv
✓ Categorical mappings saved to: ../data/processed\categorical_mappings.csv


In [21]:

# 11. Save a data dictionary for documentation
data_dict = pd.DataFrame({
    'column': df_processed.columns,
    'dtype': df_processed.dtypes.values.astype(str),
    'description': [
        'Seismic monitoring result (encoded)',
        'Seismoacoustic monitoring result (encoded)',
        'Shift type (encoded: N=dry, W=wet - higher risk)',
        'Total seismic energy from bumps',
        'Total pulses from bumps',
        'Seismic energy deviation from mean',
        'Pulse deviation from mean',
        'Hazard status (encoded)',
        'Total number of bumps',
        'Number of bumps (energy class 2)',
        'Number of bumps (energy class 3)',
        'Number of bumps (energy class 4)',
        'Number of bumps (energy class 5)',
        'Number of bumps (energy class 6)',
        'Number of bumps (energy class 7)',
        'Number of bumps (energy classes 8-9)',
        'Total seismic energy',
        'Maximum energy in single bump',
        'Target class (0=safe, 1=hazardous)',
        'Binary: any nbumps occurred',
        'Binary: any nbumps2 occurred',
        'Binary: any nbumps3 occurred',
        'Binary: any nbumps4 occurred',
        'Binary: any nbumps5 occurred',
        'Binary: any nbumps6 occurred',
        'Binary: any nbumps7 occurred',
        'Binary: any nbumps89 occurred',
        'Binary: any energy recorded',
        'Binary: maxenergy > 0',
        'Ratio of max to total energy',
        'Log-transformed energy (log1p)',
        'Shift risk indicator (1=W shift)'
    ][:len(df_processed.columns)]  # Match actual column count
})

data_dict_path = os.path.join(processed_dir, 'data_dictionary.csv')
data_dict.to_csv(data_dict_path, index=False)
print(f'✓ Data dictionary saved to: {data_dict_path}')


✓ Data dictionary saved to: ../data/processed\data_dictionary.csv
